### Topic 1 - ML Pipeline and ML tasks

- Learn and use the workflow for training and evaluating the ML pipeline.
- Create a pipeline according to our dataset and ML task.
- Fit Regression, Classification, Cluster, PCA (Principal Component Analysis), and NLP (Natural Language Processing) considering different algorithms.
- Learn and use the code to fit in one turn, multiple algorithms with hyperparameters optimisation.

Topic Objectives
- Reinforce ML pipeline concepts and the ML tasks that are covered in upcoming notebooks.
- Learn and use the workflow for training and evaluating the ML pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

### Pipeline Concept

 In previous lessons, we introduced scikit-learn which creates a Pipeline, a sequence of tasks.
* In ML, we are interested in arranging a sequence of tasks that are in line with the ML process of **data cleaning, feature engineering, feature scaling, feature selection and model**.
* In an ML pipeline, the last step is typically the model, and the preceding steps prepare the data for the model.

We import Pipeline from sklearn

In [ ]:
from sklearn.pipeline import Pipeline


 In addition, the pipeline should identify two outcomes; the training outcome and the prediction outcome. 
* For that, we use estimators as part of the pipeline steps. Two types of estimators are mainly used: predictors and transformers.
  * A predictor estimator uses methods like **.fit()** and **.predict()**. An ML model uses these methods to learn patterns from the data and is used for subsequent predictions.
  * On the other hand, the transformer estimator uses the methods **.fit()** and **.transform()** because it learns from the data and later transforms it with a better distribution. 
  


 We will demonstrate the differences between fitting models with and without a pipeline.

### Data Cleaning and Feature Engineering
In the feature-engine lesson, we studied common techniques for handling data cleaning and feature engineering tasks, such as using feature engines' built-in transformers or creating our own transformers.

In addition, we arranged this transformer in a pipeline

### Feature Scaling and Feature Selection
Once the data is cleaned and engineered, consider feature scaling and feature selection. You will be familiar with these concepts from previous lessons. Please refer to them if you need a refresh.

In this section, we will cover the practical steps of feature scaling and feature selection.

**Feature Scaling**

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> The scale of a feature is an important aspect when fitting a model. For example, algorithms like K-means clustering, linear and logistic Regression, and Neural Networks are highly affected by the scale of their features.


<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> According to Scikit-learn [documentation](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html), feature scaling can be an important preprocessing step for many machine-learning algorithms. Standardisation involves rescaling the features such that they have the properties of a standard normal distribution with a mean of zero and a standard deviation of one.
* The idea behind scaling the features is to make all features have a similar scale.


<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> We will present `StandardScaler()`, which standardises the data: it centres the variable at zero. It sets the variance to 1, by subtracting the mean from each observation and dividing by the standard deviation. It is also known as the Z-score. The documentation is [here](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
* We will cover the StandardScaler transformer in the course as a first go-to option for feature scaling. However, there are other alternatives, and you may check the [documentation](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html) to learn more. 


<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> The tradeoff of feature scaling is that the variable distribution will be slightly different. Still, we will create better conditions for the algorithm to learn the patterns and relationships in the data and generalize on unseen data.

In [ ]:
df =  sns.load_dataset('iris')
print(df.shape)
df.head()

In [ ]:
# import StandardScaler()
from sklearn.preprocessing import StandardScaler

In [ ]:
# We create a pipeline with a step called 'feature_scaling' and attach StandardScaler(). When you don't parse any variables to it, it scales all variables
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
      ("feature_scaling", StandardScaler()) 
  ])

We will apply this pipeline to the features in the train set. We will learn how to split data soon, but for now, we will manually create a train set and a test set, where each has a set for features and the target variable.
* In this dataset, features are `['sepal_length', 'sepal_width', 'petal_length', 'petal_width']` and target is `['species']`. We shuffle the data and will get the first 100 rows and set them as the train set. The remaining goes to the test.
* <img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%206%20-%20Warning.png"> There is a proper way to split a train and test set. We will cover that soon.
* The central point is to have 2 sets (Train and Test) and have features and the target separated.


Let's shuffle the data. we use `.sample(frac=1)`, the documentation link is [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html). It returns a random sample from the data.

In [ ]:
df = df.sample(frac=1)
df.head()

In [ ]:
# Split the data into training and testing sets
X_train = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']][:100]
y_train =  df[['species']][:100]
X_test =  df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']][100:]
y_test =  df[['species']][100:]
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
# Checking the shapes of the datasets (DataFrames dimensions)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
# When applying pipelines to ML, we fit the pipeline to the train set (so it will learn the parameters) and, based on this learning, transform the data on the train and test set.
pipeline.fit(X_train)
X_train_scaled = pipeline.transform(X_train)
X_test_scaled = pipeline.transform(X_test)

In [ ]:
# One caveat of using sklearn transformers is that they output NumPy arrays instead of Pandas DataFrames. You may remember that the feature-engine outputs DataFrames.
type(X_train_scaled)

In [ ]:
# So we need an additional step to convert the scaled data back to a DataFrame.
X_train_scaled = pd.DataFrame(data= X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(data= X_test_scaled, columns=X_train.columns)

In [ ]:
# Now we are fine to move on. The dataset is a DataFrame.
type(X_train_scaled)

We are now interested to see the difference in each feature before and after applying StandardScaler().
* We create a logic to loop on each feature and plot two histograms in the same plot. One shows the data distribution before applying ``StandardScaler()`` and the other after applying it.
* The blue plot is before applying, and the red is after. Note that the red histograms are centred at zero on the x-axis. You will notice the distribution may change a bit, but that is part of the tradeoff we mentioned earlier.

In [ ]:
sns.set_style('whitegrid')
for col in X_train.columns:
  fig, axes = plt.subplots(figsize=(8,5))
  sns.histplot(data=X_train, x=col, kde=True, color='b',  ax=axes)
  sns.histplot(data=X_train_scaled, x=col, kde=True,color='r', ax=axes)
  axes.set_title(f"{col}")
  axes.legend(labels=['Before Scaling', 'After Scaling'])
  plt.show()
  print("\n\n")

**Feature Selection**

The primary goal of feature selection is to have a process to select the relevant features for fitting an ML model. 

That is important since 
* Models with fewer and more relevant features are simpler to interpret.
* You reduce the chance of overfitting by removing features that may add little information or noise.
* You reduce the time needed to train the models.
* You reduce the feature space. You require less effort from the software development team to design and implement the interface (either API or dashboard) in the production environment.

This step can be seen as a combination of search techniques to look for a subset of features and an evaluation measure that scores the different feature subsets. There are a few methods for feature selection:
* Filter Method
* Wrapper Method
* Embedded Method



In the course, and as a starting point in your career, you will use the Embedded method.
* It is named the embedded method since it performs feature selection during the model training. It finds the feature subset for the algorithm that is being trained.
* The method automatically trains an ML model and then derives feature importance from it, removing non-relevant features using the derived feature importance.


For example:
* Suppose your pipeline is considering a Decision Tree algorithm in the model step. In that case, you can add before the model step a feature selection step using an embedded method considering a Decision Tree.

In [ ]:
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

X_train.head()

In [ ]:
# We are using SelectFromModel() as the method. Its documentation is found here. The argument is the algorithm you are considering in the pipeline.
from sklearn.feature_selection import SelectFromModel

We create a pipeline using a Decision Tree algorithm that contains three steps:
* `feature_scaling`: like we saw in the previous example.
* `feature_selection`: use SelectFromModel considering the same algorithm from the model step.
* `model`: uses a Decision Tree algorithm (we will get into more details in upcoming units), for now, take this step as the model step and let's use a decision tree for the example.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

pipeline = Pipeline([
      ( "feature_scaling", StandardScaler() ),
      ( "feature_selection", SelectFromModel(DecisionTreeClassifier(random_state=101)) ),
      ( "model", DecisionTreeClassifier(random_state=101) ),
  ])

pipeline

In [ ]:
# We fit the pipeline with the Train set.
pipeline.fit(X_train,y_train)

In [ ]:
# And access the feature_selection step using bracket notation as we saw in the feature-engine lesson.
pipeline['feature_selection']

That was not informative. We need to use `.get_support()` to access which features were selected by this step. 
* The output is a boolean list, where its length and order are related to the original feature space.
* For example, the train set has four features. We observe that the `feature_selection` step only selected the last list item, as it is the only one with a Boolean value of `True`. The first three features were not considered since they are `False` in the boolean list.

In [ ]:
pipeline['feature_selection'].get_support()

However, we want to know the features list that was selected, not a boolean list.
* We then use this boolean list to subset the features.
* A quick recap on the features list.

We use the boolean list to subset the previous list.
* And here we have the features that were considered important for that given dataset using that given algorithm.

In [ ]:
X_train.columns[pipeline['feature_selection'].get_support()] 

### General Workflow

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> In a practical project, you can use CRISP-DM workflow to manage project steps. In case you want a refresher on the workflow, revert to the Module Delivering Data Science projects.
* For this lesson, we will focus on the following CRISP-DM steps: data understanding, data preparation, modelling and evaluation.



<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> Therefore, when you reach the modelling phase in a project, it is assumed you have collected the data, conducted an EDA, and defined the pipeline steps.

* When modelling, for supervised learning, you will typically use an overall workflow like:
  * Split the dataset into train and test set
  * Fit the model (either a pipeline or not) 
  * Evaluate your model. If performance is not good, revisit the process, starting from collecting the data, conducting EDA etc


There are some potential small variations to this workflow, but this is the starting point we consider in your journey of modelling


 **HUGE WARNING** <img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%206%20-%20Warning.png"> 
* **Reflect** for a second on how many steps and considerations you need before fitting a model. You will be surprised that the modelling phase will take a small percentage of your time and attention in a project where a person is responsible from end to end.

* Still, this phase is critical to your project, **so let's stop the reading/talking and let's fit some models**.

